![Practical AI 101](assets/thumbnail.png)

## ⚠️ 실습 전 필수: 본인 드라이브에 사본 만들기!

상단 메뉴 **파일 → 드라이브에 사본 저장**을 클릭하세요.

사본을 만들지 않으면 작성한 코드가 저장되지 않습니다. 꼭 먼저 사본을 만든 뒤 시작하세요!

## 3차: RAG - 모델에게 기억이 아닌 근거를 주는 법

지난 실습에서는 LLM을 API로 호출하고, 필요한 도구를 스스로 골라 쓰는 **에이전트**를 만들었습니다.
그런데 도구를 잘 쓴다고 해서 모델이 모든 사실을 아는 것은 아닙니다.

<p align="center">
  <img src="assets/sejong_hallucination.png" height="360" alt="세종대왕 환각 답변 예시">
  &nbsp;&nbsp;
  <img src="assets/sejong_macbook.jpg" height="360" alt="세종대왕 맥북 프로 사건 이미지">
</p>

LLM은 위와 같이 질문에 답하려다 사건의 날짜, 이유, 신하들의 반응까지 그럴듯하게 지어낼 수 있습니다.
이렇게 사실이 아닌 내용을 자신 있게 생성하는 현상을 **환각(Hallucination)** 이라고 합니다.

오늘 아침 올라온 회사 공지나 내부 규정도 모델이 읽지 않았다면 마찬가지입니다.
모델의 기억에만 맡기면 최신 정보를 모르고, 답변의 출처도 확인하기 어렵습니다.
이 문제를 해결하기 위해 이번 주에는 **RAG**를 사용합니다.

### RAG: 먼저 찾아보고, 근거를 보며 답하기

1. **Retrieval = 가져오기**: 질문과 관련된 문서를 찾습니다.
2. **Augmented = 보태기**: 찾은 내용을 질문에 참고 자료로 붙입니다.
3. **Generation = 생성하기**: LLM이 그 근거를 읽고 답합니다.

즉, 곧바로 답하게 하지 않고 **'잠깐, 관련 문서부터 찾아봐'**라고 하는 구조입니다.
모델의 기억만 시험하던 문제를 **오픈북 시험**으로 바꾸는 셈입니다.

<p align="center">
  <img src="https://img1.daumcdn.net/thumb/R1280x0.fpng/?fname=http%3A%2F%2Ft1.daumcdn.net%2Fbrunch%2Fservice%2Fuser%2F7pqA%2Fimage%2FAU8sVcBL3EuNqO-nskDdSmCLDE0.png" width="680" alt="검색한 콘텐츠를 질문과 결합해 답변을 생성하는 RAG 프레임워크">
</p>

> 이미지·설명 참고: 김영욱, [10분만에 RAG 이해하기](https://brunch.co.kr/@ywkim36/146)

오늘은 2026년 공지를 중심으로 수집한 서울문화포털 공지사항 123건을 사용합니다.
환각을 확인하고, 문서를 검색한 뒤, 근거와 출처를 제시하는 챗봇을 직접 만들어 보겠습니다.


---
## 1. 환경 설정

준비는 네 단계입니다.

1. 라이브러리 설치: OpenRouter 호출용 `openai`, 벡터 검색용 `faiss-cpu`
2. OpenRouter API 키 입력
3. `client` 객체 만들기
4. 오늘 다룰 **2026년 서울문화포털 공지사항** 내려받기

오늘 쓰는 모델은 답변 생성용과 임베딩용 두 가지인데, 둘 다 OpenRouter를 통해 부릅니다.
그래서 설치할 것도, 발급받을 키도 하나뿐입니다.

> 💡 Colab 왼쪽 🔑(보안 비밀) 메뉴에 `OPENROUTER_API_KEY` 를 저장해 두었다면, 아래 셀이 그 값을 알아서 불러옵니다.
> 저장해 둔 값이 없으면 실행할 때 직접 입력하면 됩니다.


In [ ]:
!pip install -q --upgrade openai faiss-cpu


: 

### API 키 입력

API 키는 비밀번호이자 신용카드입니다. 코드에 직접 적어 두면 노트북을 공유하는 순간 키도 함께 새어 나가므로,
환경변수에 담아서 씁니다. 아래 셀을 실행하면 키를 물어보고, 입력한 글자는 화면에 표시되지 않습니다.


In [ ]:
import os
from getpass import getpass    # 입력한 글자를 화면에 표시하지 않는 안전한 입력 함수

if not os.environ.get("OPENROUTER_API_KEY"):
    try:
        # Colab: 왼쪽 🔑(보안 비밀) 메뉴에 OPENROUTER_API_KEY 를 저장해 뒀다면 그 값을 사용
        from google.colab import userdata
        os.environ["OPENROUTER_API_KEY"] = userdata.get("OPENROUTER_API_KEY")
    except Exception:
        # 로컬: .env 파일(OPENROUTER_API_KEY=sk-or-v1-...)이 있으면 거기서 읽기
        for path in (".env", "../.env"):
            try:
                for line in open(path):
                    key, _, value = line.strip().partition("=")
                    if value:
                        os.environ.setdefault(key, value)
                break
            except FileNotFoundError:
                pass

if not os.environ.get("OPENROUTER_API_KEY"):
    # 둘 다 없으면 직접 입력
    os.environ["OPENROUTER_API_KEY"] = getpass("OpenRouter API 키(sk-or-v1-...)를 입력하세요: ")

print("API 키 설정 완료 ✅")   # 키 자체는 절대 print 하지 않습니다

### 사용할 모델 설정

OpenRouter에 API 요청을 보낼 `client`를 만들고, 오늘 사용할 모델 두 개를 지정합니다.

- `MODEL`: 답변 생성용 모델
- `EMBED_MODEL`: 문장의 의미를 벡터로 표현해 비슷한 문서를 찾는 모델


In [ ]:
from openai import OpenAI

# OpenRouter에 요청을 보낼 API 클라이언트를 만듭니다.
client = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=os.environ["OPENROUTER_API_KEY"],
)

# 실습에서 사용할 모델을 지정합니다.
MODEL       = "nvidia/nemotron-3-ultra-550b-a55b:free"  # 답변 작성용
EMBED_MODEL = "nvidia/nemotron-3-embed-1b:free"         # 의미 검색용


### 2026년 서울문화포털 공지사항 내려받기

[서울문화포털](https://culture.seoul.go.kr/culture/bbs/B0000000/list.do?menuNo=200050)에 게시된 2026년 공지를 중심으로 123건을 미리 수집했습니다.
아래에서는 이 공지사항 파일을 읽어 들입니다.
파일 형식은 **JSONL**(JSON Lines)입니다. 한 줄에 JSON 하나씩 들어 있어서, 큰 데이터를 한 줄씩 끊어 읽기 좋습니다.

공지 하나가 딕셔너리 하나에 대응합니다.

| 키 | 내용 |
|---|---|
| `title` | 공지 제목 |
| `content` | 공지 본문 |
| `published_at` | 게시일 |
| `source_url` | 원문 링크. 나중에 답변의 **출처**로 쓰입니다 |

> 💡 아래에서 출력되는 공지를 가볍게 읽어 두세요. 잠시 후 이 공지에 대해 모델에게 질문하고,
> 모델의 답이 맞는지 여러분이 직접 채점하게 됩니다.


In [ ]:
import json

# TODO 배포 전: 로컬 상대경로를 GitHub raw URL 다운로드로 교체
with open("data/seoul_culture_notices/documents.jsonl", encoding="utf-8") as f:
    documents = [json.loads(line) for line in f]   # 한 줄 = 공지 하나

print(f"공지 {len(documents)}건")
print(f"기간: {min(d['published_at'] for d in documents)} ~ {max(d['published_at'] for d in documents)}")
print(f"본문 총 글자 수: {sum(len(d['content']) for d in documents):,}\n")

# 잠시 후 실습에 쓸 공지 하나를 미리 읽어 둡니다.
sample = next(d for d in documents if d["title"] == "2026 서울축제지도 - 여름편")
print(f"제목: {sample['title']}")
print(f"게시일: {sample['published_at']}\n")
print(sample["content"][:500])

---
## 2. 문제 제기: 모델은 모르는 것도 지어낸다

준비가 끝났으니, 방금 내려받은 공지에 대해 **문서를 주지 않고** 모델에게 물어봅시다.

모델은 이 서울문화포털 공지를 본 적이 없습니다. 대부분 2026년에 올라온 최신 글이기 때문입니다.
그러니 정직한 답은 "그 공고는 제가 알 수 없습니다"여야 합니다. 과연 그렇게 답할까요?


In [ ]:
question = "2026 서울썸머비치는 언제, 어디에서 열리고 어떤 물놀이 시설이 있어?"

response = client.chat.completions.create(
    model=MODEL,
    messages=[
        {"role": "system", "content": "모든 답변은 반드시 한국어로 작성하세요."},
        {"role": "user", "content": question},
    ],
)
print(response.choices[0].message.content)

# 💡 여러 번 실행해 보세요. 실행할 때마다 답이 달라질 수도 있습니다.

정답은 **2026년 7월 19일부터 8월 8일까지 광화문광장**에서 열리며,
**대형 수영장과 워터 슬라이드, 도심 속 모래사장과 휴게공간**이 마련된다는 것입니다.
(`2026 서울축제지도 - 여름편`에 들어 있는 실제 내용입니다.)

모델의 반응은 보통 둘 중 하나입니다.

- 그럴듯한 정보를 자신 있게 지어냅니다. 날짜와 장소, 시설까지 구체적으로 답하지만 근거가 없습니다.
- 모델이 학습한 시점까지의 정보만 알고 있어, 그 이후에 공개된 최신 공지에는 정확히 답할 수 없다고 말합니다. 정직한 답변이지만 최신 정보를 안내하는 챗봇으로는 한계가 있습니다.

어느 쪽이든 안내 챗봇으로는 실격입니다. 이것이 **환각**(hallucination) 문제입니다.
LLM은 가장 그럴듯한 다음 단어를 이어 쓰는 기계라서, 근거가 없어도 문장은 언제나 자신 있게 완성됩니다.

그렇다면 최신 공지가 나올 때마다 모델을 다시 학습시키면 되지 않을까요?
파인튜닝은 비용과 시간이 많이 들고, 정보가 바뀔 때마다 다시 학습해야 합니다. 또한 어떤 문서를 근거로 답했는지 확인하기도 어렵습니다.
그래서 모델을 계속 다시 학습시키는 대신, 질문할 때마다 최신 문서를 찾아 함께 전달하는 **RAG**를 사용합니다.

> ⭐ **오늘의 문제 정의**: 모델이 본 적 없는 문서에 대해, 지어내지 않고 출처를 밝히며 답하게 만들자.


---
## 3. 가장 단순한 해결책: 문서를 통째로 프롬프트에 넣기

해결책은 의외로 간단합니다. 모델이 문서를 모른다면, 문서를 보여주면 됩니다.

모델이 아는 것은 매 요청의 `messages` 에 담아 보낸 내용이 전부입니다.
모델에게는 지난 대화의 기억이 없어서, 프롬프트에 넣지 않은 것은 없는 것이나 마찬가지입니다.
그렇다면 공지를 통째로 `messages` 에 넣어 버리면 어떨까요?

다만 공지를 전부 넣으면 20만 자가 넘어서 모델이 한 번에 읽지 못합니다.
그래서 여기서는 **정답이 들어 있는 공지를 포함해 10건만** 넣어 보겠습니다.


In [ ]:
# 정답이 들어 있는 공지를 반드시 포함시킵니다 (없으면 넣으나 마나겠죠)
answer_doc = next(d for d in documents if d["title"] == "2026 서울축제지도 - 여름편")
some_docs = [answer_doc] + [d for d in documents if d is not answer_doc][:9]

context = "\n\n".join(f"[{d['title']}]\n{d['content']}" for d in some_docs)
print(f"프롬프트에 넣을 문서: {len(some_docs)}건 / {len(context):,}자\n")

response = client.chat.completions.create(
    model=MODEL,
    messages=[
        {"role": "system", "content": "모든 답변은 반드시 한국어로 작성하세요. 주어진 공지에 근거해서만 답하고, 공지에 없는 내용은 모른다고 답하세요."},
        {"role": "user", "content": f"다음은 서울문화포털 공지들이다.\n\n{context}\n\n질문: {question}"},
    ],
)
print(response.choices[0].message.content)
print("\n입력 토큰:", response.usage.prompt_tokens)   # 요금 미터기 ⭐

이번에는 공지에 있는 그대로 정확히 답합니다. 사실 이것이 **RAG의 본질 전부**입니다.
참고 자료를 프롬프트에 넣어 주는 것이지요.

그런데 방금 출력된 **입력 토큰 수**를 보세요. 겨우 10건을 넣었는데도 만 단위입니다.
그리고 우리가 가진 공지는 그 열 배가 넘습니다.

| 문제 | 설명 |
|---|---|
| **길이 한계** | 전부 넣으면 20만 자. 모델이 한 번에 읽을 수 있는 양(컨텍스트 길이)을 넘어선다 |
| **비용** | 입력 토큰이 곧 요금이다. 질문 하나마다 문서 전체의 값을 치르게 된다 |
| **정확도** | 관련 없는 내용이 많이 섞일수록 모델이 핵심을 놓치기 시작한다 |

무엇보다 지금은 **정답이 든 공지를 우리가 미리 골라서 넣었습니다.**
그렇게 고를 수 있었다면 애초에 챗봇이 필요하지도 않았겠지요.
그래서 실제 RAG는 여기에 한 단계를 더합니다.

> **질문과 관련 있는 문서를 자동으로 골라서 넣자.**

그렇다면 "관련 있다"를 컴퓨터는 어떻게 판단할까요? 다음 절에서 그 방법을 다룹니다.


---
## 4. 관련 문서 찾기: 키워드 검색과 의미 검색

RAG는 답변을 만들기 전에 질문과 관련된 문서를 검색합니다. 이 검색 방법은 크게 두 가지입니다.

| 검색 방법 | 무엇을 비교하나 | 장점 | 한계 |
|---|---|---|---|
| **키워드 검색** | 질문과 문서에 같은 단어가 얼마나 등장하는지 | 빠르고 결과를 이해하기 쉽다 | 표현이 다르면 같은 의미도 놓칠 수 있다 |
| **의미 검색** | 질문과 문서의 의미가 얼마나 가까운지 | 단어가 달라도 의미가 비슷한 문서를 찾는다 | 임베딩 모델이 필요하다 |

먼저 키워드를 이용하는 가장 전통적인 검색부터 시작해 보겠습니다.

### 4-1. 키워드로 찾기

여기서는 **TF-IDF**를 사용합니다. 질문과 문서에 같은 단어가 등장하면 관련성이 있다고 판단하는 방법입니다.
다만 모든 단어를 똑같이 취급하지는 않습니다.

- **TF**: 한 문서에 그 단어가 얼마나 자주 나오는지 봅니다.
- **IDF**: 여러 문서에 흔하게 나오는 단어의 중요도는 낮추고, 드물게 나오는 단어의 중요도는 높입니다.

예를 들어 `서울`, `행사`는 많은 공지에 등장하므로 결정적인 단서가 되기 어렵습니다.
반면 `서울썸머비치`처럼 일부 공지에만 나오는 단어는 원하는 문서를 찾는 강한 단서가 됩니다.
즉, TF-IDF는 **그 문서에는 자주 나오지만 다른 문서에서는 보기 힘든 단어**에 높은 점수를 줍니다.

아래 코드는 공지 제목과 본문을 TF-IDF 벡터로 바꾸고, 질문과 키워드가 가장 많이 겹치는 공지 3건을 찾습니다.


In [ ]:
%pip install -q scikit-learn

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# 제목과 본문을 합쳐 검색 대상으로 만듭니다.
notice_texts = [f"{d['title']} {d['content']}" for d in documents]

# 각 문서에서 중요한 키워드에 TF-IDF 점수를 부여합니다.
tfidf = TfidfVectorizer()
document_vectors = tfidf.fit_transform(notice_texts)

keyword_query = "서울썸머비치 광화문광장 물놀이"
query_vector = tfidf.transform([keyword_query])
scores = cosine_similarity(query_vector, document_vectors)[0]
top_indices = scores.argsort()[::-1][:3]

for rank, idx in enumerate(top_indices, start=1):
    print(f"{rank}위 ({scores[idx]:.3f}) {documents[idx]['title']}")
    print(f"   {documents[idx]['source_url']}\n")

질문에 들어 있는 `서울썸머비치`, `광화문광장`, `물놀이`가 공지에도 그대로 등장하므로 원하는 문서를 쉽게 찾을 수 있습니다.
하지만 사용자가 "도심에서 아이와 물놀이할 행사는?"처럼 **다른 표현**으로 질문하면 키워드가 겹치지 않아 검색 점수가 낮아질 수 있습니다.
이 한계를 보완하려면 단어 자체가 아니라 문장의 의미를 비교해야 합니다.

<sub>실습 참고: [scikit-learn TfidfVectorizer 문서](https://scikit-learn.org/stable/modules/generated/sklearn.feature_extraction.text.TfidfVectorizer.html)</sub>

### 4-2. 임베딩으로 의미 찾기

컴퓨터가 두 문장이 비슷한지 판단하려면, 먼저 문장을 숫자로 바꿔야 합니다.
그 변환이 **임베딩**(embedding)입니다. 문장을 좌표(벡터)로 바꾸되,
**의미가 비슷한 문장일수록 가까운 좌표**가 되도록 학습된 모델을 씁니다.

임베딩 모델에 문장을 넣으면 숫자 목록, 곧 **벡터**(vector)가 나옵니다.

![Embedding: sentence → vector](assets/SentenceEmbedding.gif)

이 숫자들은 아무렇게나 나온 값이 아니라, 문장의 **의미를 담은 좌표**입니다.
(위도, 경도) 숫자 두 개가 지도 위의 한 점을 가리키듯,
숫자 2048개는 2048차원 공간의 한 점을 가리킵니다.
그리고 모델이 그렇게 학습되어 있어서, **의미가 비슷한 문장일수록 가까운 점**에 놓입니다.
위 그림이라면 고양이가 나오는 앞의 두 문장은 서로 가까이, "The sun is bright."는 멀리 놓이는 식입니다.

그런데 좌표가 어떻게 "의미"를 담을 수 있을까요? 음식으로 감을 잡아 봅시다.
가로축을 "샌드위치다움", 세로축을 "디저트다움", 나머지 한 축을 "국물다움"이라고 정하면,
음식 하나하나를 이 3차원 공간의 한 점으로 놓을 수 있습니다.

<img src="assets/food_embedding_3d.png" alt="음식을 3차원 공간에 놓기" width="520">

<sub>그림 출처: [Google 머신러닝 단기집중과정의 임베딩 공간 문서](https://developers.google.com/machine-learning/crash-course/embeddings/embedding-space?hl=ko)의 그림입니다.</sub>

축이 하나뿐이라면 "샌드위치다움" 하나로만 음식을 줄 세워야 하지만,
축이 늘어날수록 더 많은 특성을 담을 수 있고, 그만큼 **비슷한 음식끼리는 가까운 자리에, 다른 음식은 먼 자리에** 놓이게 됩니다.
핫도그와 샤와르마는 붙어 있고, 보르시(수프)는 혼자 멀리 떨어져 있는 식입니다.

실제 임베딩 모델이 하는 일도 이것의 확장판입니다. 다만 축이 3개가 아니라 수백~수천 개이고,
각 축이 무슨 특성인지 사람이 읽어 낼 수는 없습니다. 그래도 "비슷한 것일수록 가까이 놓인다"는 성질은 그대로 유지됩니다.

이 공간이 얼마나 의미를 잘 담는지 보여 주는 유명한 그림이 있습니다.
단어들을 임베딩해서 찍어 보면, 비슷한 단어끼리 모일 뿐 아니라 **방향까지 의미를 나타냅니다**.

![word2vec: 방향도 의미를 담는다](assets/word2vec_directions.png)

<sub>그림 출처: [TensorFlow "Vector Representations of Words" 튜토리얼](https://github.com/tensorflow/docs/blob/master/site/en/r1/tutorials/representation/word2vec.md)의 그림(Mikolov et al., 2013)을 재구성한 것입니다.</sub>

king → queen 화살표와 man → woman 화살표가 나란합니다. "남성 → 여성"이라는 관계가
공간에서 하나의 방향으로 새겨진 것입니다. 동사의 시제도, 나라와 수도의 관계도 마찬가지입니다.
덕분에 `king - man + woman ≈ queen` 같은 계산까지 성립합니다.


### 직접 계산해 보기: `king - man + woman ≈ queen`

미리 학습된 영어 단어 벡터에서 `king`과 `woman`의 방향은 더하고 `man`의 방향은 빼 보겠습니다.
계산 결과와 가장 가까운 단어를 찾으면 `queen`이 나오는지 직접 확인할 수 있습니다.

> 처음 실행할 때 영어 단어 벡터 모델(`glove-wiki-gigaword-100`, 약 128MB)을 한 번 내려받습니다.


In [ ]:
%pip install -q gensim

import gensim.downloader as api

# 위키백과 등으로 미리 학습된 100차원 영어 단어 벡터를 불러옵니다.
word_vectors = api.load("glove-wiki-gigaword-100")

# king - man + woman과 가장 가까운 단어 5개를 찾습니다.
results = word_vectors.most_similar(
    positive=["king", "woman"],
    negative=["man"],
    topn=5,
)

for word, similarity in results:
    print(f"{word:10s} 유사도: {similarity:.4f}")

가장 가까운 단어로 `queen`이 출력됩니다. 단어 사이의 관계가 벡터의 **방향과 거리**에 담겨 있기 때문입니다.
이 예제는 고전적인 **단어 임베딩**의 성질을 보여 줍니다. 이어서 RAG에 사용할 **문장 임베딩**이 문장 전체의 의미를 어떻게 비교하는지 확인해 보겠습니다.

<sub>실습 참고: [Gensim 공식 KeyedVectors 문서](https://radimrehurek.com/gensim/models/keyedvectors.html)</sub>

정리하면 이렇습니다. **의미가 비슷한 문장은 벡터의 숫자들도 비슷합니다.**
숫자가 비슷하다는 것은 공간에서 두 점이 서로 가깝다는 뜻이고, 유사도가 높게 계산된다는 뜻입니다.
실제 예를 보세요. 파란 계열 상의를 가리키는 표현들인데, 단어는 전부 다르지만 숫자들이 거의 같습니다.

```json
{
  "embeddings": [
    { "word": "blue t-shirt",   "embedding": [0.72, -0.45, 0.88, 0.12, -0.65, 0.31, 0.55, 0.76] },
    { "word": "navy shirt",     "embedding": [0.71, -0.42, 0.85, 0.15, -0.68, 0.29, 0.53, 0.78] },
    { "word": "azure top",      "embedding": [0.73, -0.46, 0.87, 0.11, -0.64, 0.33, 0.56, 0.74] },
    { "word": "cobalt blouse",  "embedding": [0.75, -0.41, 0.89, 0.14, -0.67, 0.28, 0.54, 0.77] },
    { "word": "sapphire tee",   "embedding": [0.72, -0.44, 0.86, 0.13, -0.66, 0.30, 0.55, 0.76] },
    { "word": "denim shirt",    "embedding": [0.70, -0.43, 0.84, 0.16, -0.69, 0.27, 0.52, 0.79] }
  ]
}
```

검색은 이 성질을 그대로 이용합니다. 문서들을 전부 점으로 찍어 두고,
질문이 들어오면 질문도 같은 공간에 점으로 찍은 뒤,
**그 주변에서 가장 가까운 문서 점들을 집어 오면 끝**입니다.

![Query와 가장 가까운 문서](assets/semantic_search.png)

<sub>그림 출처: [sentence-transformers 문서](https://sbert.net/examples/applications/semantic-search/README.html)</sub>

"몇 명 모집해?"라는 질문과 모집 공고는 겹치는 단어가 없어도 가까운 점이 됩니다.
글자가 아니라 **의미**로 찾기 때문에, 키워드 검색이 놓치는 문서까지 찾아내는 것입니다.

### 직접 재 보기

호출 방법도 간단합니다. 답변 생성에 쓰던 `client` 를 그대로 쓰되,
`chat.completions` 대신 `embeddings` 로 보내면 글 대신 벡터가 돌아옵니다.

> 💡 우리가 쓰는 임베딩 모델은 사용 규칙상, 질문에는 `query: `, 문서에는 `passage: ` 표지를
> 앞에 붙여서 넣어야 합니다. 아래 `embed()` 가 이 표지를 대신 붙여 줍니다.
> 이 표지는 **임베딩 모델에 보낼 때만 잠깐 입히는 옷**입니다. 원본 문서는 그대로 두기 때문에,
> 나중에 검색된 문서를 LLM 프롬프트에 넣을 때는 표지가 붙지 않습니다.

두 좌표가 얼마나 가까운지는 **코사인 유사도**로 잽니다. 1에 가까울수록 비슷한 의미입니다.
벡터 길이를 1로 맞춰 두면(정규화) **내적(dot product) 한 번**으로 계산됩니다.

아래 코드에서 한국어 문장으로 직접 계산해 봅시다.


In [ ]:
import numpy as np

def embed(texts, role, batch=64):
    """문장 목록을 임베딩 벡터로 바꾼다. role: "query"(질문) 또는 "passage"(문서)"""
    texts = [f"{role}: {t}" for t in texts]      # 모델 규칙: 역할 표지를 앞에 붙인다

    vectors = []
    for i in range(0, len(texts), batch):        # 여러 문장을 한 번에 보내되, 너무 많으면 나눠 보냅니다
        response = client.embeddings.create(model=EMBED_MODEL, input=texts[i:i + batch],
                                            encoding_format="float")   # NVIDIA 모델은 base64 응답을 지원하지 않음
        vectors += [d.embedding for d in response.data]

    v = np.array(vectors, dtype="float32")
    return v / np.linalg.norm(v, axis=1, keepdims=True)    # 길이를 1로 (정규화)

sentences = [
    "미리 신청해야 하나요?",
    "사전 접수가 필요한가요?",
    "오늘 점심 메뉴 추천해줘.",
]

vectors = embed(sentences, "query")   # 셋 다 질문끼리 비교하는 것이므로 query
print("벡터 모양:", vectors.shape)     # (문장 3개, 2048차원)

# 정규화했으므로 내적 = 코사인 유사도. 1에 가까울수록 비슷한 의미입니다.
print("문장1 · 문장2 유사도:", vectors[0] @ vectors[1])   # 단어는 다르지만 의미가 같음 → 높음
print("문장1 · 문장3 유사도:", vectors[0] @ vectors[2])   # 의미가 다름 → 낮음

결과를 보세요. 문장 1과 2는 **겹치는 단어가 거의 없는데도**("미리"↔"사전", "신청"↔"접수")
유사도가 높습니다. 임베딩이 철자가 아니라 **의미**를 좌표에 새겼기 때문입니다.

두 개씩 재 보니 감이 옵니다. 이번에는 문장을 몇 개 더 모아서 **모든 쌍을 한꺼번에** 비교해 봅시다.
유사도 행렬을 그림으로 그리면, 비슷한 문장끼리 진한 블록으로 뭉치는 것이 한눈에 보입니다.


In [ ]:
import matplotlib.pyplot as plt

# Colab에는 한글 글꼴이 없어서 설치합니다 (로컬에서는 실패해도 아래에서 알아서 대체됩니다)
!apt-get -qq -y install fonts-nanum > /dev/null 2>&1
import matplotlib.font_manager as fm
for f in fm.findSystemFonts(fontpaths=["/usr/share/fonts/truetype/nanum"]):
    fm.fontManager.addfont(f)
avail = {f.name for f in fm.fontManager.ttflist}
plt.rcParams["font.family"] = next(n for n in ["NanumGothic", "AppleGothic", "sans-serif"] if n in avail or n == "sans-serif")

sentences = [
    "미리 신청해야 하나요?",
    "사전 접수가 필요한가요?",
    "아무나 참여할 수 있나요?",
    "참가 자격에 제한이 있나요?",
    "오늘 점심 메뉴 추천해줘.",
    "내일 비 올까?",
]
vectors = embed(sentences, "query")
sim = vectors @ vectors.T          # 정규화돼 있으므로 이 행렬이 곧 모든 쌍의 코사인 유사도

n = len(sentences)
fig, ax = plt.subplots(figsize=(6, 5))
im = ax.imshow(sim, cmap="YlOrRd", vmin=0, vmax=1)
ax.set_xticks(range(n), sentences, rotation=90)
ax.set_yticks(range(n), sentences)
for i in range(n):
    for j in range(n):
        ax.text(j, i, f"{sim[i, j]:.2f}", ha="center", va="center",
                color="white" if sim[i, j] > 0.6 else "black")
plt.colorbar(im)
plt.title("문장끼리의 코사인 유사도 (진할수록 비슷한 의미)")
plt.show()

그림에서 세 단계가 보입니다.

- **진한 블록**: 신청 쌍(0.78), 참가 자격 쌍(0.70) — 표현은 달라도 의미가 같은 질문끼리 묶였습니다.
- **중간 칸**: 두 쌍끼리는 0.4 안팎입니다. 서로 다른 질문이지만 넷 다 행사 참여에 관한 것이어서,
  의미가 가까운 정도가 수치에 단계적으로 나타납니다.
- **연한 칸**: 점심 메뉴, 날씨처럼 아예 다른 주제는 어떤 문장과도 낮습니다.

### 확인 실습

`sentences` 를 직접 바꿔 실험해 보세요.

- "전시회 언제까지 해?"와 "관람 기간이 어떻게 되나요?"는 얼마나 높게 나오나요?
- 같은 단어가 들어가도 의미가 다르면 어떨까요? "배가 아프다"와 "배가 항구에 들어온다"로 확인해 보세요.

문장 몇 개로 재 보는 것은 여기까지입니다. 이제 같은 방법을 실제 공지 문서에 적용해 봅시다.


---
## 5. 문서 쪼개기: 청킹(Chunking)

이제 공지를 검색할 차례인데, 그 전에 정할 것이 하나 있습니다.
**문서를 어떤 단위로 임베딩할 것인가?**

공지 하나를 통째로 벡터 하나에 담으면 어떻게 될까요? 우리 문서 중에는
7천 자가 넘는 축제 안내문도 있습니다. 그 안에 축제 수십 개의 정보가 들어 있는데 벡터 하나로 뭉개면,
"잠수교 축제" 질문과의 유사도가 나머지 내용에 묻혀 흐려집니다.

그래서 문서를 **적당한 크기의 조각(청크, chunk)** 으로 잘라 두고, 조각 단위로 검색합니다.
어떻게 자를지를 다룰 때 공통적으로 등장하는 고려 요소가 세 가지 있습니다.

| 고려 요소 | 무엇이 문제인가 | 어떻게 다루나 |
|---|---|---|
| **청크 크기** (chunk size) | 크면 여러 주제가 섞여 유사도가 흐려지고, 작으면 주제는 또렷하지만 맥락이 끊긴다 | 한 조각이 한 주제 정도를 담는 크기를 찾는다. 여기서는 상한 900자 |
| **오버랩** (overlap) | 자르는 경계에서 문장이 잘려 정보가 훼손된다 | 앞 조각의 끝부분을 다음 조각이 물고 시작하게 한다. 여기서는 120자 |
| **맥락 보존** (제목·메타데이터) | 조각만 떼어 놓으면 어느 문서에서 왔는지 알 수 없다 | 제목 같은 맥락 정보를 조각에 붙여, 조각이 혼자서도 설명되게 한다 |

900자와 120자라는 값 자체는 정답이 아니라, 다루는 문서의 길이와 구조를 보고 고르는 값입니다.

아래 `split_text` 가 이 세 가지를 구현한 전부입니다. 자를 위치는 **줄바꿈을 우선**으로 찾아서,
가능하면 문장 한가운데가 아니라 줄 경계에서 끊습니다.


<!-- manim-visual -->
### 🎬 시각 자료: 청킹과 오버랩

긴 문서가 크기 상한에 맞춰 조각으로 나뉩니다. 경계의 빨간 구간이 **오버랩**인데, 조각들을 떼어 보면 그 구간을 앞뒤 조각이 똑같이 나눠 갖고 있습니다. 덕분에 경계에서 문장이 잘려도 정보가 사라지지 않습니다.

![Chunking](assets/Chunking.gif)


In [ ]:
CHUNK_SIZE = 900    # 조각 하나의 최대 글자 수
OVERLAP    = 120    # 앞 조각의 끝 120자를 다음 조각이 물고 시작

def split_text(text, size=CHUNK_SIZE, overlap=OVERLAP):
    chunks, start = [], 0
    while start < len(text):
        end = start + size
        if end < len(text):                                  # 마지막 조각이 아니라면
            cut = text.rfind("\n", start + size - 200, end)   # 끝부분에서 줄바꿈을 찾아
            if cut > start:
                end = cut                                    # 줄 경계에서 끊는다
        chunks.append(text[start:end].strip())
        if end >= len(text):
            break
        start = end - overlap                                # ⭐ 겹치게 다음 조각 시작
    return chunks

# 공지 전체를 조각으로 나눕니다. 조각마다 제목·게시일·링크를 함께 들고 다닙니다.
chunks = []
for doc in documents:
    for piece in split_text(doc["content"]):
        chunks.append({
            "title": doc["title"],
            "published_at": doc["published_at"],
            "source_url": doc["source_url"],
            # ⭐ 임베딩·검색에 쓰이는 실제 텍스트. 제목을 앞에 붙이는 것이 핵심입니다.
            "text": f"제목: {doc['title']}\n\n{piece}",
        })

lengths = [len(c["text"]) for c in chunks]
print(f"공지 {len(documents)}건 → 청크 {len(chunks)}개")
print(f"청크 길이: 평균 {sum(lengths)//len(lengths)}자 · 최대 {max(lengths)}자")

공지 123건이 청크 300개가 되었습니다. 짧은 공지 54건은 자를 필요가 없어 한 조각 그대로이고,
가장 긴 1만 자짜리 공지는 13조각, 서두에서 이야기한 축제 안내문은 10조각으로 나뉘었습니다.

이제 임베딩은 문서당 하나가 아니라 **청크당 하나씩** 만들어집니다.
잠수교 축제 대목도 그 내용만 담은 청크가 되어 벡터 하나를 따로 가지므로,
축제 수십 개를 뭉뚱그린 벡터에 묻힐 때와 달리 "잠수교 축제" 질문과의 유사도가 또렷하게 잡힙니다.

정말 그렇게 잘렸을까요? 그 축제 안내문의 앞 두 조각을 꺼내 이음새를 직접 살펴봅시다.
앞 조각의 **끝**과 뒤 조각의 **앞**에 같은 문장이 나타나면 오버랩이 제대로 걸린 것이고,
조각마다 맨 앞에 제목이 붙어 있으면 맥락 보존까지 확인되는 것입니다.


In [ ]:
# 서두에서 이야기한 축제 안내문의 조각들을 chunks 에서 꺼냅니다.
pieces = [c["text"] for c in chunks if c["title"] == "2026 서울축제지도 - 여름편"]
first, second = pieces[0], pieces[1]
print(f"'2026 서울축제지도 - 여름편' → 조각 {len(pieces)}개\n")

print("① 앞 조각의 끝 120자")
print(repr(first[-120:]))
print("\n② 뒤 조각의 앞부분  ← 제목이 붙어 있고, 이어서 ①과 같은 문장이 다시 나옵니다")
print(repr(second[:250]))

# 💡 CHUNK_SIZE 를 300으로, OVERLAP 을 0으로 바꿔 위 셀부터 다시 실행해 보세요. 조각 수와 경계가 어떻게 달라지나요?

> 📖 [참고] 실무 청킹은 오늘 방식에서 두 가지가 더 정교해집니다.
>
> - **경계는 문서 구조로**: 문단·헤더 단위로 잘라 조각 하나가 "900자어치"가 아니라 "한 소제목 아래 내용"이 되게 합니다.
>   LangChain의 `RecursiveCharacterTextSplitter` 가 문단 → 줄바꿈 → 문장 순으로 경계를 찾는데,
>   줄 경계를 우선하는 오늘의 `split_text` 는 그중 한 단계를 구현한 셈입니다.
> - **상한은 토큰 수로**: 모델이 실제로 읽는 단위가 토큰이기 때문입니다. 다만 토큰을 세려면 토크나이저가 따로 필요해서,
>   오늘은 바로 확인되는 글자 수를 썼습니다. 900자는 우리 임베딩 모델의 토큰 한도에 여유 있게 들어갑니다.

---
## 6. 벡터 DB: 청크를 쌓아 두고, 가까운 것을 꺼내기

재료는 모두 준비됐습니다. 남은 일은 청크 300개의 벡터를 **저장해 두고**, 질문이 올 때마다
**가장 가까운 것을 찾아 꺼내는** 것입니다. 이 일을 전담하는 저장소가 **벡터 DB**입니다.
일반 DB가 "정확히 일치하는 행"을 찾아 준다면, 벡터 DB는 "이 벡터와 **가장 가까운** 벡터 k개"를 찾아 줍니다.

RAG에서 벡터 DB가 하는 일은 아래 두 국면이 전부입니다.

- **① 인덱싱 (한 번만)**: 모든 청크를 임베딩해서 벡터 DB에 쌓아 둡니다. 문서가 바뀔 때만 다시 합니다.
- **② 검색 (질문마다)**: 질문을 임베딩하고, 같은 공간에서 가장 가까운 청크 **상위 k개**를 꺼냅니다.


<!-- manim-visual -->
### 🎬 시각 자료: 벡터 DB의 두 국면

인덱싱은 한 번만 일어납니다. 청크마다 벡터 하나가 만들어져 인덱스에 점으로 쌓입니다. 검색은 질문마다 일어납니다. 질문도 같은 공간의 점이 되고, 가장 가까운 점 k개가 찾아지면 그 청크들이 프롬프트로 갑니다.

![VectorDB](assets/VectorDB.gif)


이 벡터 DB 역할로 오늘은 **FAISS**(Meta 개발)를 씁니다. FAISS는 서버를 따로 띄우는
데이터베이스가 아니라 **파이썬에서 바로 불러 쓰는 벡터 검색 라이브러리**라서,
설치 한 줄이면 노트북 안에서 실습하기에 알맞습니다. 쓰는 법도 세 줄이면 끝납니다.

| 코드 | 하는 일 |
|---|---|
| `faiss.IndexFlatIP(차원수)` | 빈 인덱스를 만든다. `IP` 는 Inner Product(내적)이고, 정규화된 벡터에서는 내적이 곧 코사인 유사도다 |
| `index.add(벡터들)` | 청크 벡터를 인덱스에 저장한다 |
| `index.search(질문벡터, k)` | 유사도 상위 k개의 (점수, 번호)를 돌려준다 |

> 📖 [참고] `IndexFlatIP` 의 Flat은 저장된 벡터를 전부 비교하는 전수조사 방식을 뜻합니다.
> 청크가 수백 개면 충분하지만, 수백만 개가 되면 IVF나 HNSW 같은 **근사 검색 인덱스**로 바꿔 속도를 확보합니다.
> Chroma, Pinecone 같은 벡터 DB 서비스도 내부 원리는 같습니다. 오늘 배우는 구조가 그대로 실무의 구조입니다.


In [ ]:
import faiss

# ① 인덱싱: 청크를 전부 임베딩해서 FAISS 인덱스에 저장 (한 번만 하면 됩니다)
print(f"청크 {len(chunks)}개를 임베딩하는 중... (30초쯤 걸립니다)")
chunk_vectors = embed([c["text"] for c in chunks], "passage")   # ⭐ 청크는 '찾히는 쪽'
print("청크 벡터 모양:", chunk_vectors.shape)         # (청크 수, 2048)

index = faiss.IndexFlatIP(chunk_vectors.shape[1])     # 내적(=코사인 유사도) 기반 인덱스
index.add(chunk_vectors)
print("인덱스에 저장된 벡터 수:", index.ntotal)

# ② 검색 함수: 질문 → 유사도 상위 k개 청크
def search(question, k=3):
    q = embed([question], "query")                    # ⭐ 질문은 '찾는 쪽'
    scores, ids = index.search(q, k)                  # 상위 k개의 (유사도, 청크 번호)
    return [(chunks[i], float(s)) for i, s in zip(ids[0], scores[0])]

# 검색 테스트 — 아직 답변 생성 모델은 등장하지 않습니다. 순수한 '찾기'입니다.
for chunk, score in search(question):
    print(f"({score:.3f}) {chunk['title'][:50]}")


1등으로 나온 공지를 보세요. 자원봉사자 질문에 정확히 그 모집 공고가 뽑혔나요?
3절에서는 우리가 정답 문서를 손으로 골라 넣었지만, 이제는 **컴퓨터가 알아서 찾아냅니다.**

### 확인 실습

여러 질문으로 검색해 보고, 1등 공지가 정말 답을 담고 있는지 눈으로 확인하세요.

```python
search("DDP에서 버스킹 하려면 어떻게 신청해?")
search("여름에 아이랑 갈 만한 프로그램 있어?")
search("공연장 대관 신청은 어디서 해?")
```

> 💡 일부러 **공지에 없는 질문**("서울시 지하철 요금이 얼마야?")도 해 보세요.
> 그래도 유사도가 가장 높은 청크가 뽑혀 나옵니다. 검색은 가장 비슷한 것을 골라 줄 뿐,
> 거기에 답이 들어 있는지까지는 판단하지 못합니다.
> 그래서 다음 절에서 모델에게 "문서에 없으면 모른다고 답하라"는 지시를 함께 줍니다.


---
## 7. 검색한 청크를 프롬프트에 넣기

이제 두 조각을 이어 붙입니다.

```
질문 → ① search() 로 관련 청크 k개 검색 → ② 청크를 프롬프트에 담아 질문과 함께 전송 → ③ 근거 있는 답변
```

3절과 다른 점은 하나뿐입니다. 문서를 **우리가 골라 넣는** 대신 **검색이 골라 준다**는 것입니다.


질문 하나가 지나가는 전체 과정을 한 그림으로 보면 이렇습니다.
오른쪽 절반은 우리가 5~6절에서 만든 인덱싱과 검색이고, 왼쪽 아래가 이번 절에서 만들 부분입니다.
같은 질문에 RAG가 없으면 모른다고 하거나 지어내지만(without RAG), RAG가 있으면
검색된 청크를 근거로 답합니다(with RAG). 암기 시험이 오픈북 시험으로 바뀌는 순간입니다.

![RAG 파이프라인 전체](assets/rag_pipeline_ko.png)

<sub>그림 출처: [HuggingFace Cookbook 한국어판](https://huggingface.co/learn/cookbook/ko/advanced_ko_rag)(김하림 번역)</sub>

아래 `ask_rag()` 가 이 그림의 왼쪽 아래 부분을 코드로 옮긴 것입니다. `search()` 로 관련 청크를 찾고(①),
`build_context()` 로 하나의 문자열로 묶어 질문과 함께 보냅니다(②). 이때 system 프롬프트에는
규칙 두 가지를 함께 적습니다.

- **근거 제한**: "주어진 공지에 근거해서만 답하고, 없으면 모른다고 하라." 환각을 막는 핵심 규칙입니다.
- **출처 표시**: "공지 제목과 링크를 밝혀라." 사용자가 원문을 직접 확인할 수 있어야 하니, 안내 챗봇에서는 필수입니다.


In [ ]:
SYSTEM_PROMPT = """너는 서울문화포털 공지 안내 챗봇이다.
- 주어진 공지에 근거해서만 답한다.
- 공지에 없는 내용은 지어내지 말고 모른다고 답한다.
- 답 끝에 근거로 쓴 공지 제목과 링크를 밝힌다."""

def build_context(results):
    """검색된 청크들을 프롬프트에 넣을 하나의 문자열로 만든다"""
    return "\n\n".join(
        f"[{c['title']}] (게시일 {c['published_at']}, 링크 {c['source_url']})\n{c['text']}"
        for c, _ in results
    )

def ask_rag(question, k=3):
    # ① 검색: 질문과 가장 관련 있는 청크 k개를 찾는다
    results = search(question, k)
    context = build_context(results)

    # ② 생성: 검색된 청크를 근거로 붙여 질문한다
    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": f"공지:\n{context}\n\n질문: {question}"},
        ],
    )

    print("🔍 검색된 근거:")
    for c, score in results:
        print(f"  ({score:.3f}) {c['title'][:50]}")
    print()
    return response.choices[0].message.content

print(ask_rag(question))

2절에서 모델이 지어내던 **바로 그 질문**에, 이제 출처까지 달아 정확히 답합니다.
오늘 거쳐 온 세 가지 방식을 나란히 놓고 보세요.

| | 2절 (문서 없이) | 3절 (문서를 통째로) | 지금 (RAG) |
|---|---|---|---|
| 답변 | 그럴듯하게 **지어내거나** 모른다고 함 | 정확함 | 정확함 + **출처 링크** |
| 문서 선택 | 없음 | **사람이 손으로** 골라 넣음 | **검색이 자동으로** 골라 줌 |
| 입력 토큰 | 질문뿐 | 넣은 문서 전체 (만 단위) | 검색된 청크 k개뿐 |
| 문서가 늘어나면 | 달라질 것 없음 (어차피 모름) | 컨텍스트 한계에 부딪힘 | 입력은 그대로, 인덱스만 커짐 |

정말 완성됐는지는 질문을 더 던져 봐야 알겠지요. 아래 셀에서 직접 실험해 보세요.
여러 공지를 엮어야 답할 수 있는 질문과, 공지에 없는 질문까지 던져 보면 좋습니다.


In [ ]:
# 💡 다른 질문으로도 실습해 보세요.
print(ask_rag("DDP에서 버스킹을 하려면 어떻게 신청해?"))

# print(ask_rag("서울시 문화상은 어떤 분야에 주고, 추천은 언제까지야?"))
# print(ask_rag("여름에 아이랑 갈 만한 프로그램 추천해줘"))     # 여러 공지를 엮어 답할까요?
# print(ask_rag("서울시 지하철 요금이 얼마야?"))                # 공지에 없는 질문 — 모른다고 답하는지 보세요

🎉 **RAG 완성입니다.** 앞에서 본 그림의 흐름 그대로, 오늘 우리가 만든 코드를 단계별로 짚으면 이렇습니다.

| 국면 | 단계 | 우리가 만든 코드 |
|---|---|---|
| **인덱싱** (한 번만) | 청킹: 공지 123건 → 청크 300개 | `split_text()` (5절) |
| | 임베딩: 청크마다 벡터 하나 | `embed(…, "passage")` (4절) |
| | 저장: 벡터를 인덱스에 쌓기 | `index.add()` (6절) |
| **검색·생성** (질문마다) | 검색: 질문과 가장 가까운 청크 k개 찾기 | `search()` (6절) |
| | 증강: 찾은 청크를 프롬프트에 담기 | `build_context()` (7절) |
| | 생성: 근거를 읽고 답변, 출처 표시 | `ask_rag()` (7절) |


---
## 8. 연습문제 🎯

문제는 세 개이고, 하나로 이어지는 이야기입니다.

| 문제 | 하는 일 | 여러분이 할 일 | 핵심 |
|---|---|---|---|
| **1** | 페르소나 기능 추가하기 | 빈칸 1개 ✏️ | **RAG = 검색 결과를 프롬프트에 넣는 것** |
| **2** | "모르면 모른다"는 안내원 만들기 | 프롬프트 실험 👀 | 환각을 막는 행동 규칙 |
| **3** | **내 문서**로 나만의 챗봇 만들기 | 문서 교체 ✏️ | 문서만 갈아 끼우면 파이프라인은 그대로 |

### 문제 1. 페르소나 기능 추가하기 (✏️ 빈칸 1개)

7절의 `ask_rag` 를 다시 만납니다. 이번 버전(`ask_rag_v2`)에는 문제 2를 위해
**system 프롬프트를 바꿔 끼우는 기능**이 추가되어 있습니다.

빈칸은 한 곳, **검색 결과(`context`)를 프롬프트에 넣는 자리**입니다.

하필 이 자리를 비워 둔 이유가 있습니다. 오늘 배운 내용 전체가 이 한 줄로 요약되기 때문입니다.

> ⭐ **RAG는 모델을 바꾸는 기술이 아니라, 프롬프트에 근거를 넣어 주는 기술입니다.**
> 검색이 아무리 정확해도, 찾아낸 청크를 프롬프트에 넣지 않으면 모델은 여전히 아무것도 모릅니다.

> ⚠️ 빈칸을 채우기 전에 실행하면 모델이 근거 없이 답합니다. 2절에서 본 환각으로 되돌아가는 셈이지요.
> 채우기 전과 후를 일부러 비교해 보는 것도 좋은 실험입니다.


In [ ]:
def ask_rag_v2(question, system_prompt, k=3):
    results = search(question, k)
    context = build_context(results)

    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": system_prompt},
            # TODO ✏️: 검색 결과를 프롬프트에 넣으세요. ________ 자리에 들어갈 변수 이름은?
            {"role": "user", "content": f"공지:\n{________}\n\n질문: {question}"},
        ],
    )
    return response.choices[0].message.content

# 기본 규칙으로 테스트 — 7절과 비슷한 답이 나오면 성공!
basic_rule = "주어진 공지에 근거해서만 답한다. 공지에 없는 내용은 모른다고 답한다."
print(ask_rag_v2("서울도서관 자원봉사자 활동 기간은 언제까지야?", system_prompt=basic_rule))

<details>
<summary> 문제 1 정답 보기</summary>

```python
{"role": "user", "content": f"공지:\n{context}\n\n질문: {question}"}
```

`search()` 가 찾아온 청크 묶음(`context`)을 질문 앞에 붙여서 보내는 것, 이 한 줄이 RAG의 전부입니다.
검색(Retrieval)한 것을 프롬프트에 증강(Augmented)해서 생성(Generation)한다는 이름 그대로입니다.

</details>


### 문제 2. "모르면 모른다"고 답하는 안내원 만들기 (프롬프트 실험 👀)

이번에는 채울 코드가 없습니다. 완성된 프롬프트를 실행해 보고 **직접 바꿔 보는** 문제입니다.

RAG의 마지막 안전장치는 검색도 임베딩도 아닌 **system 프롬프트의 행동 규칙**입니다.
아래 프롬프트에는 실제 안내 챗봇들이 쓰는 규칙 세 가지가 들어 있습니다.

    ① 주어진 공지에 근거해서만 답하기. 환각을 막는 핵심 규칙입니다.
    ② 공지에 없으면 지어내지 말고 "공지에서 찾지 못했습니다. 서울문화포털에서 직접 확인해 주세요"라고
       안내하기. 모른다고 말할 용기와, 사람에게 넘기는 출구를 함께 마련하는 규칙입니다.
    ③ 게시일과 링크를 함께 밝히기. 이미 지난 공고일 수도 있으니 사용자가 직접 확인할 수 있어야 합니다.

특히 ②를 시험해 보는 것이 이 문제의 핵심입니다. **답이 공지에 없는 질문**을 던졌을 때
지어내지 않고 버티는지 확인해 보세요.


In [ ]:
guide_prompt = """너는 서울문화포털 안내원이다.
- 주어진 공지에 근거해서만 답한다.
- 공지에 없는 내용은 절대 지어내지 않고, '공지에서 찾지 못했습니다. 서울문화포털(culture.seoul.go.kr)에서 직접 확인해 주세요.'라고 답한다.
- 답 끝에 근거 공지의 제목·게시일·링크를 밝힌다. 접수 기간이 이미 지났을 수 있다면 그 점도 알려준다."""

# ✅ 성공 기준 1: 공지에 있는 질문 → 정확한 답 + 제목·게시일·링크
print(ask_rag_v2("서울시 문화상 추천은 언제까지 접수해?", system_prompt=guide_prompt))
print()
# ✅ 성공 기준 2: 공지에 없는 질문 → "공지에서 찾지 못했습니다..." (지어내면 실패!)
print(ask_rag_v2("서울시 따릉이 이용 요금이 얼마야?", system_prompt=guide_prompt))

# ✏️ 프롬프트를 직접 바꿔 실험해 보세요. 프롬프트에는 정답이 없습니다!
#    "초등학생도 이해할 수 있게, 친근한 말투로 답한다." 를 추가하면?
#    규칙 ②를 지우면, 공지에 없는 질문에서 무슨 일이 일어날까요?

### 문제 3. 내 문서로 나만의 챗봇 만들기

마지막 과제입니다. `MY_DOCS` 에 **여러분만의 문서**를 넣으세요. 무엇이든 좋습니다.

- 수강 중인 과목의 강의계획서, 동아리 회칙, 아르바이트 근무 매뉴얼
- 좋아하는 게임의 공략 메모, 자취방 계약서 조항, 여러분이 직접 쓴 글

조건은 하나뿐입니다. 모델이 모를 만한 내용일수록 결과가 재미있습니다. (개인정보는 넣지 마세요!)

아래 셀은 5절과 6절의 파이프라인을 **그대로** 다시 돌립니다. 새로 배울 것은 없습니다.
문서가 바뀌어도 청킹, 임베딩, FAISS 인덱싱 코드는 **한 글자도 바뀌지 않는다**는 것이
오늘의 마지막 요점입니다.


In [ ]:
MY_DOCS = [
    {"title": "내 문서 1", "content": """(여기에 문서 내용을 붙여 넣으세요.
길면 알아서 여러 청크로 잘립니다.)"""},
    {"title": "내 문서 2", "content": """(두 번째 문서. 개수는 늘려도 됩니다.)"""},
]

# ── 아래는 5~6절의 파이프라인 그대로입니다 ──
chunks = []
for doc in MY_DOCS:
    doc.setdefault("published_at", "-")
    doc.setdefault("source_url", "-")
    for piece in split_text(doc["content"]):
        chunks.append({**doc, "text": f"제목: {doc['title']}\n\n{piece}"})

chunk_vectors = embed([c["text"] for c in chunks], "passage")
index = faiss.IndexFlatIP(chunk_vectors.shape[1])
index.add(chunk_vectors)
print(f"청크 {index.ntotal}개 인덱싱 완료 ✅\n")

# ✅ 성공 기준: 내 문서에만 있는 내용을 물었을 때 정확히 답하면, 나만의 RAG 챗봇 완성!
my_prompt = "주어진 문서에 근거해서만 답한다. 문서에 없는 내용은 '문서에서 찾지 못했습니다'라고 답한다."
print(ask_rag_v2("여러분의 문서에 대한 질문을 여기에 쓰세요", system_prompt=my_prompt))

# ⚠️ 서울문화포털 챗봇으로 되돌리려면 5절과 6절의 셀을 다시 실행하면 됩니다.


---
## 정리

### 오늘 배운 것

| 주제 | 핵심 |
|---|---|
| **환각** | 모델은 모르는 것도 그럴듯하게 지어낸다. 모른다고 말해 주지 않는다 |
| **RAG** | 참고 자료를 찾아 프롬프트에 넣어 주는 것. 모델을 다시 학습시키는 것이 아니다 |
| **임베딩** | 문장을 의미의 좌표(벡터)로 바꾼다. 의미가 비슷하면 좌표가 가깝다 |
| **코사인 유사도** | 벡터를 정규화해 두면 내적 한 번으로 계산된다 |
| **청킹** | 크기 상한과 오버랩, 제목 붙이기. 경계에서 정보가 잘리지 않게 한다 |
| **FAISS** | `IndexFlatIP` 에 벡터를 쌓고 `search()` 로 상위 k개를 꺼낸다. Chroma와 Pinecone도 같은 원리다 |
| **근거 제한 프롬프트** | "공지에 근거해서만, 없으면 모른다고." 환각을 막는 마지막 안전장치다 |

### 반드시 기억할 세 가지

1. **모델은 모르는 것도 지어낸다.** 그래서 근거를 쥐여 주는 RAG가 필요합니다.
2. **RAG의 품질은 검색의 품질이 결정한다.** 엉뚱한 청크를 찾아오면 생성이 아무리 좋아도 답은 틀립니다.
3. **RAG는 검색 결과를 프롬프트에 넣는 것이다.** 특별한 마법이 아니라 `messages` 를 다루는 방법의 응용입니다.

### 추가 학습 과제

오늘 만든 챗봇을 실무 수준으로 키울 때 만나게 될 키워드들입니다.

- **근사 검색 인덱스** (IVF, HNSW): 청크가 수백만 개일 때 FAISS를 빠르게 쓰는 방법
- **하이브리드 검색** (BM25 + 임베딩): 키워드 검색과 의미 검색을 함께 쓰는 방법
- **리랭킹 (Reranking)**: 넓게 찾아 온 후보를 더 정밀한 모델로 다시 정렬하는 방법
- **쿼리 재작성 (Query Rewriting)**: 검색하기 전에 질문을 검색에 유리한 형태로 다듬는 방법
- **RAG와 에이전트의 결합**: 함수 호출로 검색을 도구처럼 등록해 두면, 모델이 필요할 때만 스스로 검색합니다

수고하셨습니다! 👏
